In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

In [2]:
df=pd.read_csv('job_dataset.csv')

In [3]:
df.head(2)

,JobID,Title,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,Keywords
0,NET-F-001,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Framework; .NET Core f...,Assist in coding and debugging applications; L...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
1,NET-F-002,.NET Developer,Fresher,0-1,C#; .NET Framework basics; ASP.NET; Razor; HTM...,Write simple C# programs under guidance; Suppo...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...


## Handling Missing value

In [3]:
df=df.dropna(subset=['Title'])

In [4]:
df['Title'].shape

(1067,)

## Data Cleaning

In [5]:
import re 
import string

def year_cleaning(value):
    value=str(value)
    match=re.search(r'\d',value)
    if match:
        return int(match[0])
    return None

def text_cleaning(text):
    text=str(text)
    text=text.lower()
    text=re.sub(r'\s','',text)
    # text=text.strip()
    return text


In [6]:
df['Title']=df['Title'].apply(text_cleaning)

In [7]:
df['YearsOfExperience']=df['YearsOfExperience'].apply(year_cleaning)

In [8]:
df['YearsOfExperience'].value_counts()

YearsOfExperience
0    434
5    170
3     95
6     81
1     80
4     74
2     55
7     41
8     23
9     14
Name: count, dtype: int64

In [ ]:
df['Keywords']=df['Keywords'].apply(text_cleaning)

## Feature Engineering

In [14]:
from sklearn.preprocessing import  MultiLabelBinarizer


>convert keyword feature into binary features

In [11]:
df['keywords_list']=df['Keywords'].str.split(';')

In [12]:
df['keywords_list']

0       [.net, c#, asp.netmvc, entityframework, sqlser...
1       [.net, c#, asp.netmvc, entityframework, sqlser...
2       [.net, c#, asp.netmvc, sqlserver, entityframew...
3         [.net, c#, sqlserver, entityframework, asp.net]
4       [.net, c#, asp.netmvc, entityframework, sqlser...
                              ...                        
1063    [angular, node.js, express.js, django, sql, re...
1064    [react, node.js, express.js, sql, mongodb, doc...
1065    [vue.js, node.js, express.js, sql, redis, dock...
1066    [react, node.js, express.js, mongodb, docker, ...
1067    [angular, node.js, express.js, sql, redis, doc...
Name: keywords_list, Length: 1067, dtype: object

In [15]:
mlb_keyword=MultiLabelBinarizer()

In [16]:
keywords_encoded=mlb_keyword.fit_transform(df['keywords_list'])

In [17]:
mlb_keyword.classes_

array(['.net', '.netcore', '3ddesign', '3dgraphics', '3dmodeling',
       'a/btesting', 'accessibility', 'activedirectory', 'adaptability',
       'adobecreativesuite', 'adobeframemaker', 'adobeillustrator',
       'adobephotoshop', 'adobexd', 'advancedanalytics', 'agile',
       'agile/scrum', 'agiledocumentation', 'agileleadership', 'agileux',
       'ai', 'ai-assistedcoding', 'aianalytics', 'aidashboards',
       'aidesigntools', 'aiethics', 'aiframeworks', 'aimarketingtools',
       'aimodeloptimization', 'airflow', 'aiseotools', 'aitools',
       'aiwritingtools', 'ajax', 'analytics', 'android', 'androidndk',
       'androidsdk', 'androidstudio', 'angular', 'animation', 'ansible',
       'apachespark', 'apiarchitecture', 'apidevelopment',
       'apidocumentation', 'apiintegration', 'apis', 'apitesting',
       'appdevelopment', 'appium', 'appoptimization', 'appstorerelease',
       'appstoresubmission', 'ar/vrdesign', 'architecture',
       'architecturedesign', 'arcore', 'arkit'

In [18]:
keywords_df=pd.DataFrame(
    keywords_encoded,
    columns=mlb_keyword.classes_,
    index=df.index
)

In [19]:
keywords_df

,.net,.netcore,3ddesign,3dgraphics,3dmodeling,a/btesting,accessibility,activedirectory,adaptability,adobecreativesuite,...,wordpress,workflowautomation,workflowimprovement,workflowmanagement,workflowobservation,workflowoptimization,xcode,xml,xunit,zerotrust
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1063,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1064,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1065,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1066,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [23]:
keywords_df['3dgraphics'].value_counts()

3dgraphics
0    1052
1      15
Name: count, dtype: int64

In [20]:
keywords_df['.net'].value_counts()

.net
0    1057
1      10
Name: count, dtype: int64

In [21]:
X = pd.concat([
    df['Title'],
    df['ExperienceLevel'],
    df['YearsOfExperience'],
    keywords_df,
], axis=1)

## Feature Encoding 

In [22]:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder

In [23]:
lb=LabelEncoder()

In [26]:
df['Title_encod']=lb.fit_transform(df['Title'])

In [25]:
ohe=OneHotEncoder(sparse_output=False)
encoded=ohe.fit_transform(df[['ExperienceLevel']])

In [27]:
encoded_df = pd.DataFrame(
    encoded,
    columns=ohe.get_feature_names_out(['ExperienceLevel']),
    index=df.index
)

In [29]:
df['Title_encod'].shape

(1067,)

In [40]:
new_df=pd.concat([
    df['Title_encod'],
    encoded_df,
    df['YearsOfExperience'],
    keywords_df,
], axis=1)

In [41]:
new_df=new_df.sort_values('Title_encod')

In [47]:
X=new_df.drop('Title_encod',axis=1)
y=new_df['Title_encod']

In [48]:
X.select_dtypes(include='object').columns

Index([], dtype='object')

## Train-Test Split

In [51]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [59]:
from sklearn.metrics import accuracy_score

# Model Training

## Logistic Regression

In [34]:
from sklearn.linear_model import LogisticRegression

lg = LogisticRegression(max_iter=1000)

lg.fit(X_train, y_train)

y_pred_lg = lg.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_lg))

Accuracy: 0.7897196261682243


## DecisionTreeClassifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

print(accuracy_score(y_test, y_pred_dt))

0.780373831775701


## RandomForestClassifier

In [38]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100,random_state=42)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print(accuracy_score(y_test, y_pred_rf))

0.7897196261682243


## XGBClassifier

## AdaBoostClassifier

In [91]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(
    n_estimators=100,
    random_state=42
)

ada.fit(X_train, y_train)

y_pred = ada.predict(X_test)

In [92]:
print(accuracy_score(y_test, y_pred))

0.08878504672897196


In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()

nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)
print(accuracy_score(y_test, y_pred_nb))

0.7383177570093458


## KNeighborsClassifier

In [94]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(
    n_neighbors=5
)

knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)
print(accuracy_score(y_test, y_pred_knn))

0.7336448598130841


## GradientBoostingClassifier

In [60]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    random_state=42
)

gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)

print(accuracy_score(y_test, y_pred_gb))

0.8037383177570093


## Filter the data

In [95]:
pred='.netdeveloper'

In [96]:
pred_role=df[df["Title"]==pred]

In [97]:
pred_role.head(1)

,JobID,Title,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,Keywords,keywords_list,Title_encod
0,NET-F-001,.netdeveloper,Fresher,0,C#; VB.NET basics; .NET Framework; .NET Core f...,Assist in coding and debugging applications; L...,.net;c#;asp.netmvc;entityframework;sqlserver;l...,"[.net, c#, asp.netmvc, entityframework, sqlser...",0


## Find Similar Candidates

In [98]:
Combined= (pred_role["Keywords"] +' '+pred_role['ExperienceLevel']+' '+pred_role["Title"])

In [99]:
Combined

0     .net;c#;asp.netmvc;entityframework;sqlserver;l...
1     .net;c#;asp.netmvc;entityframework;sqlserver;r...
2     .net;c#;asp.netmvc;sqlserver;entityframework;g...
3     .net;c#;sqlserver;entityframework;asp.net Fres...
4     .net;c#;asp.netmvc;entityframework;sqlserver F...
5     .net;c#;asp.net;razor;sqlserver;unittesting Fr...
6     .net;c#;asp.netmvc;sqlserver Fresher .netdevel...
7     .net;c#;asp.netmvc;entityframework;linq;sqlser...
8       .net;c#;asp.net;sqlserver Fresher .netdeveloper
9     .net;c#;asp.netmvc;entityframework;sqlserver;g...
10    .netcore;c#;asp.netmvc;microservices;azure;ent...
11    .netcore;c#;asp.netmvc;solid;azuredevops;entit...
12    .netcore;c#;microservices;docker;kubernetes;az...
13    .netcore;c#;asp.netmvc;webapi;signalr;entityfr...
14    .netcore;c#;asp.netmvc;blazor;angular;sqlserve...
15    .netcore;c#;asp.netmvc;restapis;docker;azurede...
16    .netcore;c#;webapi;grpc;sqlserver;azure Experi...
17    .netcore;c#;microservices;kubernetes;entit

## TF-IDF Vectorization

In [100]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=500)

X_tfidf = tfidf.fit_transform(Combined)

## Similarity Search

In [101]:
from sklearn.metrics.pairwise import cosine_similarity

In [102]:
user_text = "Machine Learning Deep Learning Python TensorFlow ML Engineer"


In [103]:
user_vector = tfidf.transform([user_text])

In [105]:
similarity = cosine_similarity(
    user_vector,
    X_tfidf
)

In [106]:
similarity

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.]])

In [107]:
best_index = similarity.argmax()

In [108]:
best_index

np.int64(0)

## Return Responsibility

In [109]:
recommended_responsibility = \
df.iloc[best_index]["Responsibilities"]

In [110]:
recommended_responsibility

'Assist in coding and debugging applications; Learn and apply .NET Framework and Core fundamentals; Support team in building ASP.NET MVC web applications; Write basic SQL queries and work with Entity Framework; Collaborate with peers to solve issues; Participate in code reviews for learning; Follow best practices in coding; Work with version control (Git)'

# Cross Validation

In [62]:
from sklearn.model_selection import cross_val_score

In [ ]:
from sklearn.linear_model import LogisticRegression

lg = LogisticRegression(max_iter=1000)

scores = cross_val_score(
    lg,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print(scores)

d:\DataScience and AI programming\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


[0.82242991 0.8317757  0.82629108 0.84976526 0.84037559]


In [47]:
print("Mean Accuracy:", scores.mean())

Mean Accuracy: 0.8341275064718529


In [48]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42
)

scores = cross_val_score(
    rf,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print(scores)
print(scores.mean())

d:\DataScience and AI programming\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


[0.8317757  0.8364486  0.83098592 0.84976526 0.84037559]
0.8378702119257602


In [49]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()

scores = cross_val_score(
    dt,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print(scores.mean())

d:\DataScience and AI programming\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


0.8069501118862709


In [63]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier()

scores = cross_val_score(
    knn,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print(scores.mean())

d:\DataScience and AI programming\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


0.7909920582686147


In [66]:
from sklearn.naive_bayes import MultinomialNB
mnb = MultinomialNB()

scores = cross_val_score(
    mnb,
    X,
    y,
    cv=5,
    scoring='accuracy'
)
print(scores)
print(scores.mean())

d:\DataScience and AI programming\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


[0.80373832 0.82242991 0.80751174 0.81690141 0.75117371]
0.8003510157518319


# Hypermeter Tunning

In [51]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

params = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    params,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)

d:\DataScience and AI programming\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
d:\DataScience and AI programming\venv\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
25 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
25 fits failed with the following error:
Traceback (most recent call last):
  File "d:\DataScience and AI programming\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\DataScience a

{'C': 1, 'solver': 'lbfgs'}
0.8452631578947368


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

params = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10]
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(),
    param_distributions=params,
    n_iter=20,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print(random_search.best_params_)
print(random_search.best_score_)

d:\DataScience and AI programming\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


{'n_estimators': 500, 'min_samples_split': 2, 'max_depth': None}


In [112]:
print(random_search.best_score_)

0.8440866873065016
